# Organize Libraries and Imports

In [20]:
import requests
import json
from collections import defaultdict
import json
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta
from requests.exceptions import RequestException

import pandas as pd

## Get Familiar with API Endpoints

In [1]:
def get_dhs_sub_agencies():
    # DHS Top-tier Agency Code is '070'
    url = "https://api.usaspending.gov/api/v2/agency/070/sub_agency/?fiscal_year=2026"

    response = requests.get(url)
    if response.status_code == 200:
        sub_agencies = response.json().get('results', [])
        print(f"{'Sub-Agency Name':<50} | {'Obligations'}")
        print("-" * 70)
        for sa in sub_agencies:
            name = sa.get('name')
            obs = sa.get('total_obligations', 0)
            print(f"{name:<50} | ${obs:,.2f}")
    else:
        print(f"Error: {response.status_code}")

get_dhs_sub_agencies()

Sub-Agency Name                                    | Obligations
----------------------------------------------------------------------
U.S. Customs and Border Protection                 | $19,216,739,873.18
U.S. Coast Guard                                   | $4,188,524,381.44
Federal Emergency Management Agency                | $4,143,003,557.82
U.S. Immigration and Customs Enforcement           | $3,799,096,976.56
Office of Procurement Operations                   | $1,905,170,032.01
Transportation Security Administration             | $417,746,362.66
U.S. Citizenship and Immigration Services          | $413,949,822.05
Federal Law Enforcement Training Center            | $260,191,028.70
U.S. Secret Service                                | $117,716,874.08
Office of the Inspector General                    | $9,181,413.66


In [2]:
def get_dhs_fy25_fixed():
    search_url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"

    # Prefix mapping for DHS Components based on Award ID
    # This acts as a backup when the API returns 'None'
    DHS_PREFIXES = {
        "HSTS": "TSA (Transportation Security Admin)",
        "HSFE": "FEMA (Federal Emergency Mgmt Agency)",
        "HSCG": "U.S. Coast Guard",
        "HSBP": "CBP (Customs & Border Protection)",
        "HSSC": "USCIS (Citizenship & Immigration)",
        "HSIG": "Office of Inspector General",
        "70Z":  "U.S. Coast Guard (Modern Prefix)",
        "70B":  "CBP (Modern Prefix)",
    }

    payload = {
        "filters": {
            "time_period": [{"start_date": "2024-10-01", "end_date": "2025-09-30"}],
            "agencies": [{"type": "awarding", "tier": "toptier", "name": "Department of Homeland Security"}],
            "naics_codes": {"require": ["518210", "541511", "541512", "513210"]},
            "award_type_codes": ["A", "B", "C", "D"]
        },
        "fields": [
            "Award ID",
            "Recipient Name",
            "Award Amount",
            "Awarding Agency",
            "Awarding Sub Tier Agency"
        ],
        "limit": 100
    }

    try:
        response = requests.post(search_url, json=payload)
        if response.status_code == 200:
            results = response.json().get('results', [])

            component_totals = defaultdict(float)
            component_counts = defaultdict(int)

            for award in results:
                # 1. Try to get name from API field
                agency = award.get('Awarding Sub Tier Agency')

                # 2. If API returned None, use the Award ID prefix logic
                if not agency or agency == "None":
                    award_id = award.get('Award ID', "")
                    # Match the first 4 characters of the ID
                    prefix = award_id[:4]
                    agency = DHS_PREFIXES.get(prefix, "DHS - Other/Headquarters")

                amount = award.get('Award Amount') or 0.0
                component_totals[agency] += float(amount)
                component_counts[agency] += 1

            # Print Summary
            print(f"\n{'DHS COMPONENT (RESOLVED)':<45} | {'AWARDS':<7} | {'TOTAL OBLIGATED'}")
            print("-" * 80)
            for agency, total in sorted(component_totals.items(), key=lambda x: x[1], reverse=True):
                print(f"{agency[:43]:<45} | {component_counts[agency]:<7} | ${total:,.2f}")

        else:
            print(f"Error: {response.status_code}")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    get_dhs_fy25_fixed()


DHS COMPONENT (RESOLVED)                      | AWARDS  | TOTAL OBLIGATED
--------------------------------------------------------------------------------
CBP (Customs & Border Protection)             | 2       | $617,988,126.68
USCIS (Citizenship & Immigration)             | 8       | $280,746,569.28
DHS - Other/Headquarters                      | 47      | $211,936,149.53
TSA (Transportation Security Admin)           | 5       | $71,998,203.87
U.S. Coast Guard                              | 28      | $62,344,523.90
FEMA (Federal Emergency Mgmt Agency)          | 9       | $34,140,981.26
Office of Inspector General                   | 1       | $1,770,707.10


# Create Query for Award and Transaction Data

## Set Consistent Query Parameters

In [5]:
# Fiscal Year 2021 (10/01/2020) through 04/20/2026
start = '2020-10-01'
end = '2026-04-20'

# Define all cabinets plus several independent agencies
target_departments = [
    "Department of Homeland Security", "Department of Justice",
    "Department of Defense",
    "Department of State", "Department of the Treasury",
    "Department of Energy", "Department of Commerce",
    "Department of Health and Human Services", "Department of Agriculture",
    "Department of the Interior", "Department of Transportation",
    "Securities and Exchange Commission", "Commodity Futures Trading Commission"
]

award_type_codes = ["A", "B", "C", "D"]

# Technology Only NAICS codes of interest
naics_codes = ["518210", "541511", "541519", "541512", "513210", "511210", "541512", "334111"]

# Award Spending - Use to Get Description of Contract

## Get Award Data
- This returns > 1,000,000 records when run for the 5 year period across all agencies. Result is saved as a csv

In [36]:
# Documentation: https://github.com/fedspendingtransparency/usaspending-api/blob/master/usaspending_api/api_contracts/contracts/v2/search/spending_by_award.md
search_url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
all_award_results = []

# Define your total range
start_dt = datetime.strptime(start, "%Y-%m-%d")
end_dt = datetime.strptime(end, "%Y-%m-%d")

for dept in target_departments:
    print('\n', '=' * 25, f"DEPARTMENT: {dept}")
    
    # Initialize current chunk start
    current_chunk_start = start_dt
    
    while current_chunk_start < end_dt:
        # Create a 6-month chunk (Quarterly)
        current_chunk_end = current_chunk_start + relativedelta(months=6) - relativedelta(days=1)
        
        # Ensure we don't overshoot the final end date
        if current_chunk_end > end_dt:
            current_chunk_end = end_dt
            
        s_str = current_chunk_start.strftime("%Y-%m-%d")
        e_str = current_chunk_end.strftime("%Y-%m-%d")
        
        print(f"--- Fetching: {s_str} to {e_str} ---")
        page = 1

        while True:
            payload = {
                "filters": {
                    "time_period": [{"start_date": s_str, "end_date": e_str}],
                    "agencies": [{"type": "awarding", "tier": "toptier", "name": dept}],
                    "award_type_codes": award_type_codes,
                    "naics_codes": {"require": naics_codes},
                },
                "fields": ["Award ID", "generated_internal_id", "Recipient Name", "Recipient DUNS Number", "recipient_id",
                           "Award Amount", "Awarding Agency", "Awarding Sub Agency", "Funding Agency", "Funding Sub Agency",
                           "Place of Performance City Code", "Place of Performance State Code", "Place of Performance Country Code",
                           "Place of Performance Zip5", "Last Modified Date", "Base Obligation Date",
                           "Start Date", "End Date", "Description"],
                "limit": 100,
                "page": page,
                "order": "desc",
                "sort": "Award Amount"
            }

            try:
                response = requests.post(search_url, json=payload, timeout=60)
                response.raise_for_status()
                data = response.json()

                results = data.get('results', [])
                if not results:
                    break

                all_award_results.extend(results)
                print(f"Page {page}: Added {len(results)} records. Total: {len(all_award_results)}")

                if len(results) < 100:
                    break

                page += 1
                time.sleep(0.5) # Politeness delay

            except Exception as e:
                print(f"Error on {s_str} page {page}: {e}")
                print("Retrying in 5 seconds...")
                time.sleep(5)
                # This will loop back and try the same page again
                continue 

        # Move to the next 3-month chunk
        current_chunk_start += relativedelta(months=3)

print(f"\nDone! Combined total: {len(all_award_results)} awards.")


 ========================= DEPARTMENT: Department of Homeland Security
--- Fetching: 2020-10-01 to 2021-03-31 ---
Page 1: Added 100 records. Total: 100
Page 2: Added 100 records. Total: 200
Page 3: Added 100 records. Total: 300
Page 4: Added 100 records. Total: 400
Page 5: Added 100 records. Total: 500
Page 6: Added 100 records. Total: 600
Page 7: Added 100 records. Total: 700
Page 8: Added 100 records. Total: 800
Page 9: Added 100 records. Total: 900
Page 10: Added 100 records. Total: 1000
Page 11: Added 100 records. Total: 1100
Page 12: Added 100 records. Total: 1200
Page 13: Added 100 records. Total: 1300
Page 14: Added 100 records. Total: 1400
Page 15: Added 100 records. Total: 1500
Page 16: Added 100 records. Total: 1600
Page 17: Added 100 records. Total: 1700
Page 18: Added 100 records. Total: 1800
Page 19: Added 100 records. Total: 1900
Page 20: Added 100 records. Total: 2000
Page 21: Added 100 records. Total: 2100
Page 22: Added 100 records. Total: 2200
Page 23: Added 100 reco

### Write output to CSV

In [45]:
df_awards_v3 = pd.DataFrame(all_award_results)
# df_awards_v3.to_csv('df_all_award_results_v3.csv', index=False)

## Get Transaction Data
- Transaction Spending Gives Picture of Yearly Spend
- Award spending could include all spending since inception of contract
- Cell below takes A LONG time to run (~600,000 transactions)

In [34]:
# Config
# Documentation: https://github.com/fedspendingtransparency/usaspending-api/blob/master/usaspending_api/api_contracts/contracts/v2/search/spending_by_transaction.md
search_url = "https://api.usaspending.gov/api/v2/search/spending_by_transaction/"
start_dt = datetime.strptime(start, "%Y-%m-%d")
end_dt = datetime.strptime(end, "%Y-%m-%d")

all_tx_results = []

for dept in target_departments:
    print('\n', '=' * 25, f"DEPARTMENT: {dept}")
    
    current_chunk_start = start_dt
    
    while current_chunk_start < end_dt:
        # Define the 6-month window
        current_chunk_end = current_chunk_start + relativedelta(months=6) - relativedelta(days=1)
        if current_chunk_end > end_dt:
            current_chunk_end = end_dt
            
        s_str = current_chunk_start.strftime("%Y-%m-%d")
        e_str = current_chunk_end.strftime("%Y-%m-%d")
        
        print(f"\n--- Range: {s_str} to {e_str} ---")
        page = 1

        while True:
            payload = {
                "filters": {
                    "time_period": [{"start_date": s_str, "end_date": e_str}],
                    "agencies": [{"type": "awarding", "tier": "toptier", "name": dept}],
                    "award_type_codes": award_type_codes,
                    "naics_codes": {"require": naics_codes},
                },
                "fields": [
                    "Award ID", "internal_id", "generated_internal_id", "Recipient Name", "Action Date",
                    "Award Type", "Mod", "Transaction Amount", "Transaction Description", 
                    "Awarding Sub Agency", "Awarding Agency", "PSC", "NAICS", "Action Date"
                ],
                "limit": 100,
                "page": page,
                "sort": "Transaction Amount",
                "order": "desc"
            }

            success = False
            for attempt in range(5):
                try:
                    # 'timeout' is key to prevent hanging on a dead connection
                    response = requests.post(search_url, json=payload, timeout=45)
                    response.raise_for_status()
                    data = response.json()
                    success = True
                    break 
                except (RequestException, Exception) as e:
                    wait = 2 ** attempt
                    print(f"      [Retry {attempt+1}] Error: {e}. Waiting {wait}s...")
                    time.sleep(wait)

            if not success:
                print(f"      CRITICAL: Failed to retrieve {s_str} Page {page}. Skipping chunk.")
                break

            results = data.get('results', [])
            if not results:
                break

            all_tx_results.extend(results)
            print(f"      Page {page}: +{len(results)} tx (Total: {len(all_tx_results)})")

            # Check if we've reached the end of this date chunk
            if len(results) < 100:
                break

            page += 1
            time.sleep(0.2) # Small cooldown for the API

        # Move to the next 3-month block
        current_chunk_start = current_chunk_end + relativedelta(days=1)

print(f"\nCompleted! Final count: {len(all_tx_results)}")


 ========================= DEPARTMENT: Department of Homeland Security

--- Range: 2020-10-01 to 2021-03-31 ---
      Page 1: +100 tx (Total: 100)
      Page 2: +100 tx (Total: 200)
      Page 3: +100 tx (Total: 300)
      Page 4: +100 tx (Total: 400)
      Page 5: +100 tx (Total: 500)
      Page 6: +100 tx (Total: 600)
      Page 7: +100 tx (Total: 700)
      Page 8: +100 tx (Total: 800)
      Page 9: +100 tx (Total: 900)
      Page 10: +100 tx (Total: 1000)
      Page 11: +100 tx (Total: 1100)
      Page 12: +100 tx (Total: 1200)
      Page 13: +100 tx (Total: 1300)
      Page 14: +100 tx (Total: 1400)
      Page 15: +100 tx (Total: 1500)
      Page 16: +100 tx (Total: 1600)
      Page 17: +100 tx (Total: 1700)
      Page 18: +100 tx (Total: 1800)
      Page 19: +100 tx (Total: 1900)
      Page 20: +100 tx (Total: 2000)
      Page 21: +100 tx (Total: 2100)
      Page 22: +100 tx (Total: 2200)
      Page 23: +100 tx (Total: 2300)
      Page 24: +100 tx (Total: 2400)
      Page 25: +1

### Create Transactions DataFrame and save as CSV

In [35]:
# Create df Transactions DataFrame
df_transactions_v3 = pd.DataFrame(all_tx_results)
# df_transactions_v3.to_csv('df_transactions_v3.csv', index=False)

In [44]:
pd.read_csv('df_transactions_v3.csv').info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 665939 entries, 0 to 665938
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Unnamed: 0               665939 non-null  int64  
 1   Award ID                 665939 non-null  object 
 2   internal_id              665939 non-null  int64  
 3   generated_internal_id    665939 non-null  object 
 4   Recipient Name           665939 non-null  object 
 5   Action Date              665939 non-null  object 
 6   Award Type               665939 non-null  object 
 7   Mod                      665939 non-null  object 
 8   Transaction Amount       665939 non-null  float64
 9   Transaction Description  665769 non-null  object 
 10  Awarding Sub Agency      665939 non-null  object 
 11  Awarding Agency          665939 non-null  object 
 12  PSC                      665939 non-null  object 
 13  NAICS                    665939 non-null  object 
dtypes: f

# Read in DataFrames (once saved) and Aggregate

In [15]:
# Awards Data
dfa = pd.read_csv('df_all_award_results.csv')

# Transactions Data
df_transactions.read_csv('df_transactions.csv')

In [22]:
dfa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1075860 entries, 0 to 1075859
Data columns (total 13 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1075860 non-null  int64  
 1   internal_id            1075860 non-null  int64  
 2   Award ID               1075860 non-null  object 
 3   generated_internal_id  1075860 non-null  object 
 4   Recipient Name         1075860 non-null  object 
 5   Award Amount           1075860 non-null  float64
 6   Awarding Agency        1075860 non-null  object 
 7   Awarding Sub Agency    1075860 non-null  object 
 8   Start Date             1075853 non-null  object 
 9   End Date               1075860 non-null  object 
 10  Description            1075209 non-null  object 
 11  awarding_agency_id     1075860 non-null  int64  
 12  agency_slug            1075860 non-null  object 
dtypes: float64(1), int64(3), object(9)
memory usage: 106.7+ MB


In [32]:
len(dfa['Award ID'].unique())

301738

In [21]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 665939 entries, 0 to 665938
Data columns (total 8 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Award ID               665939 non-null  object 
 1   Recipient Name         665939 non-null  object 
 2   Action Date            665939 non-null  object 
 3   Transaction Amount     665939 non-null  float64
 4   Awarding Agency        665939 non-null  object 
 5   PSC                    665939 non-null  object 
 6   generated_internal_id  665939 non-null  object 
 7   internal_id            665939 non-null  int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 40.6+ MB


In [33]:
len(df_transactions['Award ID'].unique())
# df_transactions.head()

301787

In [16]:
# Merge to df_awards
df_awards_abbrev = df_awards[['generated_internal_id', 'Description', 'Award Amount', 'Start Date', 'End Date']].copy()
df_transactions_description = pd.merge(df_transactions, df_awards_abbrev, how = 'left', on = 'generated_internal_id')

In [19]:
df_awards_abbrev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1075860 entries, 0 to 1075859
Data columns (total 5 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   generated_internal_id  1075860 non-null  object 
 1   Description            1075419 non-null  object 
 2   Award Amount           1075860 non-null  float64
 3   Start Date             1075853 non-null  object 
 4   End Date               1075860 non-null  object 
dtypes: float64(1), object(4)
memory usage: 41.0+ MB


In [17]:
df_transactions_description.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4613206 entries, 0 to 4613205
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Award ID               object 
 1   Recipient Name         object 
 2   Action Date            object 
 3   Transaction Amount     float64
 4   Awarding Agency        object 
 5   PSC                    object 
 6   generated_internal_id  object 
 7   internal_id            int64  
 8   Description            object 
 9   Award Amount           float64
 10  Start Date             object 
 11  End Date               object 
dtypes: float64(2), int64(1), object(9)
memory usage: 422.4+ MB


### Write Pulled USA Spending Data to CSV

In [ ]:
path = 'df_all_transactions.csv'
# df_transactions_description.to_csv(path, index=False)